03_feature_engineering.ipynb
+ yeni feature'ların doğrulanması
+ feature importance analizleri
+ feature karşılaştırmaları

**Baseline modelimiz:**
ROC-AUC = 0.7724

>Bu sonucu hangi feature'lar üretti?

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

print(project_root)

c:\Users\ASUS\creditguard-ai


In [3]:
import joblib
import pandas as pd

from src.preprocessing.prepare_dataset import (
    prepare_dataset
)

from src.models.feature_importance import (
    get_feature_importance
)

In [ ]:
df = prepare_dataset(
    "../data/processed/train_feature_store.parquet"
)

print(df.shape)

(307511, 149)


+ Önce : 150 kolon
+ Sonra : 149 kolon (SK_ID_CURR çıkarıldı)


In [ ]:
X = df.drop(columns=["TARGET"])

feature_names = X.columns

len(feature_names)



148

In [ ]:
model = joblib.load(
    "../artifacts/models/lgbm_baseline.pkl"
)

importance_df = get_feature_importance(
    model,
    feature_names
)

importance_df.head(20)

,feature,importance
39,EXT_SOURCE_1,462
40,EXT_SOURCE_2,419
41,EXT_SOURCE_3,410
122,credit_term,308
130,annuity_credit_ratio,305
18,DAYS_ID_PUBLISH,269
15,DAYS_BIRTH,240
134,bureau_total_credit,237
17,DAYS_REGISTRATION,232
139,bureau_debt_credit_ratio,206


## 📊 Öne Çıkan Özellikler (Feature Importance)

Model eğitimi sonucunda, kendi ürettiğimiz (engineered) özellikler ve harici veri kaynaklarından türetilen agregasyonlar arasında en yüksek ayırt edici güce sahip, **ilk 20'ye giren kritik özellikler** aşağıda listelenmiştir:

|  Sıra  | Özellik Adı (Feature Name)    | Açıklama                                                                  |
| :----: | :---------------------------- | :------------------------------------------------------------------------ |
|  **1** | `credit_term`                 | Kredi tutarının taksit tutarına oranı; kredi vadesi hakkında bilgi sağlar |
|  **2** | `annuity_credit_ratio`        | Taksit tutarının toplam kredi tutarına oranı                              |
|  **3** | `bureau_total_credit`         | Kredi Kayıt Bürosu'ndaki toplam kredi hacmi                               |
|  **4** | `bureau_debt_credit_ratio`    | Toplam borcun toplam krediye oranı                                        |
|  **5** | `bureau_total_debt`           | Kredi Kayıt Bürosu'ndaki toplam mevcut borç                               |
|  **6** | `age_years`                   | Müşterinin yaşı (yıl bazında)                                             |
|  **7** | `credit_income_ratio`         | Kredi tutarının müşterinin gelirine oranı                                 |
|  **8** | `annuity_income_ratio`        | Taksit tutarının müşterinin gelirine oranı                                |
|  **9** | `prev_avg_credit_amount`      | Önceki başvurulardaki ortalama kredi tutarı                               |
| **10** | `prev_avg_application_amount` | Önceki başvurularda talep edilen ortalama kredi miktarı                   |
| **11** | `refusal_rate`                | Geçmiş kredi başvurularında reddedilme oranı                              |

### Sonuç

Feature engineering çalışmaları sonucunda oluşturulan değişkenlerin önemli bir bölümü modelin en etkili değişkenleri arasına girmiştir. Özellikle:

* Gelir–kredi ilişkisini ölçen oranlar (`credit_income_ratio`, `annuity_income_ratio`)
* Kredi geçmişini özetleyen bureau agregasyonları (`bureau_total_credit`, `bureau_total_debt`, `bureau_debt_credit_ratio`)
* Geçmiş kredi başvuru davranışlarını özetleyen değişkenler (`prev_avg_credit_amount`, `prev_avg_application_amount`, `refusal_rate`)

model tarafından güçlü risk sinyalleri olarak kullanılmıştır.

Bu sonuçlar, yalnızca ham başvuru verilerinin değil, müşteri geçmişinden türetilen davranışsal ve finansal göstergelerin de kredi temerrüt tahmininde önemli katkı sağladığını göstermektedir.


In [ ]:
importance_df[
    importance_df["feature"] == "SK_ID_CURR"
]

,feature,importance


In [ ]:
importance_df[
    importance_df["feature"] == "EXT_SOURCE_1"
]

,feature,importance
39,EXT_SOURCE_1,462


In [1]:
import pandas as pd

pos = pd.read_csv(
    "../data/raw/POS_CASH_balance.csv"
)

print(pos.shape)

(10001358, 8)


In [4]:
from src.features.build_pos_cash_features import (
    build_pos_cash_features
)

pos_features = (
    build_pos_cash_features(pos)
)

print(pos_features.shape)

(337252, 9)


**previous_features**    → 338857 müşteri

**installment_features** → 339587 müşteri

**bureau_features**    → 305811 müşteri

**pos_features**       → 337252 müşteri

In [5]:
pos_features.head()

,SK_ID_CURR,pos_record_count,pos_avg_dpd,pos_max_dpd,pos_avg_dpd_def,pos_max_dpd_def,pos_active_contracts,pos_completed_contracts,pos_avg_future_installments
0,100001,9,0.777778,7,0.777778,7,7.0,2.0,1.444444
1,100002,19,0.000000,0,0.000000,0,19.0,NaN,15.000000
2,100003,28,0.000000,0,0.000000,0,26.0,2.0,5.785714
3,100004,4,0.000000,0,0.000000,0,3.0,1.0,2.250000
4,100005,11,0.000000,0,0.000000,0,9.0,1.0,7.200000


In [6]:
pos_features.describe().T

,count,mean,std,min,25%,50%,75%,max
SK_ID_CURR,337252.0,278163.132678,102877.889290,100001.0,189046.75,278241.500000,367320.250000,456255.000000
pos_record_count,337252.0,29.655445,24.531971,1.0,12.00,22.000000,39.000000,295.000000
pos_avg_dpd,337252.0,4.296271,59.717229,0.0,0.00,0.000000,0.000000,2622.078431
pos_max_dpd,337252.0,15.294106,151.343806,0.0,0.00,0.000000,0.000000,4231.000000
pos_avg_dpd_def,337252.0,0.225470,13.554576,0.0,0.00,0.000000,0.000000,1740.554455
pos_max_dpd_def,337252.0,1.473355,32.337266,0.0,0.00,0.000000,0.000000,3595.000000
pos_active_contracts,337034.0,27.151916,22.723824,1.0,11.00,20.000000,36.000000,271.000000
pos_completed_contracts,300240.0,2.480959,3.253642,1.0,1.00,2.000000,3.000000,86.000000
pos_avg_future_installments,337224.0,9.176876,6.501034,0.0,5.00,6.989411,11.666667,60.000000


In [2]:
import joblib

from src.preprocessing.prepare_dataset import (
    prepare_dataset
)

from src.models.feature_importance import (
    get_feature_importance
)

df = prepare_dataset(
    "../data/processed/train_feature_store.parquet"
)

X = df.drop(columns=["TARGET"])

feature_names = X.columns

model = joblib.load(
    "../artifacts/models/lgbm_baseline.pkl"
)

importance_df = get_feature_importance(
    model,
    feature_names
)

importance_df.head(30)

,feature,importance
39,EXT_SOURCE_1,398
40,EXT_SOURCE_2,356
41,EXT_SOURCE_3,352
162,pos_avg_future_installments,344
130,annuity_credit_ratio,265
122,credit_term,231
18,DAYS_ID_PUBLISH,224
154,total_payment_amount,223
15,DAYS_BIRTH,209
139,bureau_debt_credit_ratio,200


### Ürettiğimiz Feature'lardan İlk 30'a Girenler
**Application**
Feature | Importance |
-----------|----------------|
annuity_credit_ratio|	265
credit_term	|231
annuity_income_ratio	|155
age_years	|125
credit_income_ratio|	110



**Bureau**
Feature|Importance |
-----------|---------------|
bureau_debt_credit_ratio|	200
bureau_total_credit	|191
bureau_total_debt	|144
bureau_active_loans	|139

**Previous Application**
Feature|Importance |
-----------|---------------|
prev_avg_application_amount|	112

**Installments**
Feature|Importance |
-----------|---------------|
total_payment_amount|	223
late_payment_ratio	|166
installment_count	|131
avg_payment_ratio	|130
avg_days_late  |	111

**POS Cash**
Feature	 | Importance
-----------|---------------|
pos_avg_future_installments|	344
pos_active_contracts|	122

## Feature Engineering Impact

Feature engineering çalışmaları sonucunda oluşturulan müşteri davranışı ve kredi geçmişi özellikleri model performansını önemli ölçüde artırmıştır.

Özellikle aşağıdaki özellikler en yüksek önem skorlarına ulaşmıştır:

- pos_avg_future_installments
- annuity_credit_ratio
- credit_term
- total_payment_amount
- bureau_debt_credit_ratio
- bureau_total_credit
- late_payment_ratio
- annuity_income_ratio
- pos_active_contracts
- avg_payment_ratio

Bu özellikler müşterinin mevcut yükümlülük seviyesini, ödeme disiplinini ve geçmiş kredi davranışını temsil etmektedir.

In [5]:
import pandas as pd

credit_card = pd.read_csv(
    "../data/raw/credit_card_balance.csv"
)

from src.features.build_credit_card_features import (
    build_credit_card_features
)

credit_card_features = (
    build_credit_card_features(
        credit_card
    )
)

print(credit_card_features.shape)

credit_card_features.head()



(103558, 11)


,SK_ID_CURR,cc_record_count,cc_avg_balance,cc_max_balance,cc_avg_credit_limit,cc_utilization_ratio,cc_avg_payment,cc_total_payment,cc_avg_dpd,cc_max_dpd,cc_avg_drawings
0,100006,6,0.000000,0.00,270000.000000,0.000000,0.000000,0.000,0.000000,0,0.000000
1,100011,74,54482.111149,189000.00,164189.189189,0.302678,4520.067568,334485.000,0.000000,0,2432.432432
2,100013,96,18159.919219,161420.22,131718.750000,0.115301,6817.172344,654448.545,0.010417,1,5953.125000
3,100021,17,0.000000,0.00,675000.000000,0.000000,0.000000,0.000,0.000000,0,0.000000
4,100023,8,0.000000,0.00,135000.000000,0.000000,0.000000,0.000,0.000000,0,0.000000


In [6]:


credit_card_features.describe().T

,count,mean,std,min,25%,50%,75%,max
SK_ID_CURR,103558.0,278381.457956,102779.519683,100006.000000,189536.25,278649.000000,367690.000000,4.562500e+05
cc_record_count,103558.0,37.083683,33.483627,1.000000,10.00,22.000000,75.000000,1.920000e+02
cc_avg_balance,103558.0,69973.192455,107537.810551,-2930.232558,0.00,24997.602995,96997.746023,9.286863e+05
cc_max_balance,103558.0,142297.932093,171325.542574,0.000000,0.00,96107.175000,194612.546250,1.505902e+06
cc_avg_credit_limit,103558.0,207320.669739,190229.277094,0.000000,82500.00,149000.000000,267500.000000,1.350000e+06
cc_utilization_ratio,102445.0,0.320573,0.324382,-0.084848,0.00,0.241313,0.583274,2.138790e+00
cc_avg_payment,103558.0,10266.210653,21915.296936,0.000000,0.00,3986.601378,11874.107601,1.591837e+06
cc_total_payment,103558.0,281422.754426,462156.168193,0.000000,0.00,118969.852500,393300.000000,3.007314e+07
cc_avg_dpd,103558.0,4.107206,44.341025,0.000000,0.00,0.000000,0.000000,1.635685e+03
cc_max_dpd,103558.0,16.401871,141.966150,0.000000,0.00,0.000000,0.000000,3.260000e+03


**cc_utilization_ratio** - mean = 0.32

> Müşteriler ortalama limitlerinin %32'sini kullanıyor.

In [3]:
df = prepare_dataset(
    "../data/processed/train_feature_store.parquet"
)

print(df.shape)

(307511, 174)


In [4]:
X = df.drop(columns=["TARGET"])

feature_names = X.columns

len(feature_names)

173

In [5]:
model = joblib.load(
    "../artifacts/models/lgbm_baseline.pkl"
)

importance_df = get_feature_importance(
    model,
    feature_names
)

importance_df.head(40)

,feature,importance
39,EXT_SOURCE_1,376
40,EXT_SOURCE_2,354
41,EXT_SOURCE_3,336
162,pos_avg_future_installments,308
154,total_payment_amount,232
130,annuity_credit_ratio,223
122,credit_term,216
18,DAYS_ID_PUBLISH,197
15,DAYS_BIRTH,181
139,bureau_debt_credit_ratio,168


### Credit Card Sprint Sonucu

En önemli bulgu:

| Feature                |        Importance |
| ---------------------- | ----------------: |
| `cc_utilization_ratio` |           **133** |
| `cc_avg_drawings`      |            **84** |
| `cc_avg_balance`       | İlk 40'a giremedi |
| `cc_avg_payment`       | İlk 40'a giremedi |
| `cc_total_payment`     | İlk 40'a giremedi |
| `cc_avg_dpd`           | İlk 40'a giremedi |


### Şu Ana Kadarki En Güçlü Engineered Feature'lar

**Model artık ağırlıklı olarak şu gruplardan öğreniyor:**

**Application**

+ credit_term

+ annuity_credit_ratio

+ credit_income_ratio

+ annuity_income_ratio

+ age_years

**Bureau**

+ bureau_total_credit

+ bureau_total_debt

+ bureau_debt_credit_ratio

+ bureau_active_loans

**Previous Application**

+ approval_rate

+ refusal_rate

+ prev_avg_application_amount

**Installments**

+ total_payment_amount

+ late_payment_ratio

+ avg_days_late

+ installment_count

+ avg_payment_ratio

**POS Cash**

+ pos_avg_future_installments

+ pos_record_count

+ pos_active_contracts

**Credit Card**

+ cc_utilization_ratio

+ cc_avg_drawings


In [4]:
bureau = pd.read_csv(
    "../data/raw/bureau.csv"
)

bureau_balance = pd.read_csv(
    "../data/raw/bureau_balance.csv"
)

In [5]:
from src.features.build_bureau_balance_features import (
    build_bureau_balance_features
)

bb_features = (
    build_bureau_balance_features(
        bureau_balance,
        bureau
    )
)

print(bb_features.shape)

bb_features.head()

c:\Users\ASUS\creditguard-ai\src\features\build_bureau_balance_features.py:13: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace(


(305811, 6)


,SK_ID_CURR,bb_record_count,bb_history_length,bb_late_ratio,bb_severe_late_ratio,bb_max_status
0,100001,172.0,23.571429,0.007519,0.0,1.0
1,100002,110.0,28.250000,0.255682,0.0,1.0
2,100003,0.0,NaN,NaN,NaN,NaN
3,100004,0.0,NaN,NaN,NaN,NaN
4,100005,21.0,6.000000,0.000000,0.0,0.0


In [6]:
bb_features.describe().T

,count,mean,std,min,25%,50%,75%,max
SK_ID_CURR,305811.0,278047.300091,102849.568343,100001.0,188878.500000,277895.000000,367184.50,456255.0
bb_record_count,305811.0,79.067597,148.193152,0.0,0.000000,0.000000,101.00,2791.0
bb_history_length,134542.0,34.573140,17.663957,0.0,21.666667,33.545455,45.50,96.0
bb_late_ratio,134542.0,0.016113,0.049311,0.0,0.000000,0.000000,0.01,1.0
bb_severe_late_ratio,134542.0,0.002334,0.025632,0.0,0.000000,0.000000,0.00,1.0
bb_max_status,134542.0,0.514821,0.932831,0.0,0.000000,0.000000,1.00,5.0


In [7]:
from src.preprocessing.prepare_dataset import (
    prepare_dataset
)

from src.models.feature_importance import (
    get_feature_importance
)

import joblib

df = prepare_dataset(
    "../data/processed/train_feature_store.parquet"
)

X = df.drop(columns=["TARGET"])

feature_names = X.columns

model = joblib.load(
    #"../artifacts/models/lgbm_tuned.pkl"
    "../artifacts/models/lgbm_optuna_cv.pkl"
)

importance_df = get_feature_importance(
    model,
    feature_names
)

importance_df.head(50)

,feature,importance
41,EXT_SOURCE_3,276
162,pos_avg_future_installments,258
122,credit_term,254
130,annuity_credit_ratio,246
39,EXT_SOURCE_1,239
40,EXT_SOURCE_2,227
134,bureau_total_credit,215
154,total_payment_amount,197
15,DAYS_BIRTH,190
8,AMT_GOODS_PRICE,178


In [8]:
importance_df[
    importance_df["feature"]
    .str.contains("bb_")
]

,feature,importance
173,bb_record_count,73
174,bb_history_length,65
182,bb_months_since_last_late,42
175,bb_late_ratio,34
180,bb_last_12m_late_ratio,20
177,bb_max_status,9
176,bb_severe_late_ratio,8
181,bb_recent_late_count,5
179,bb_recent_max_status,1
178,bb_last_status,0


In [4]:
from src.preprocessing.prepare_dataset import (
    prepare_dataset
)

from src.models.feature_importance import (
    get_feature_importance
)

import joblib

df = prepare_dataset(
    "../data/processed/train_feature_store.parquet"
)

X = df.drop(columns=["TARGET"])

feature_names = X.columns

model = joblib.load(
    "../artifacts/models/lgbm_optuna_cv.pkl"
)

importance_df = get_feature_importance(
    model,
    feature_names
)

In [5]:
print(len(feature_names))
print(len(model.feature_importances_))

194
194


In [6]:
importance_df[
    importance_df["feature"]
    .str.contains("bureau")
]

,feature,importance
140,bureau_avg_credit_per_loan,167
134,bureau_total_credit,155
139,bureau_debt_credit_ratio,126
135,bureau_total_debt,110
143,bureau_credit_per_active_loan,108
144,bureau_debt_per_active_loan,92
142,bureau_active_loan_ratio,76
136,bureau_active_loans,72
141,bureau_avg_debt_per_loan,65
137,bureau_closed_loans,64


In [2]:
import joblib

from src.preprocessing.prepare_dataset import (
    prepare_dataset
)

from src.models.feature_importance import (
    get_feature_importance
)

In [3]:
df = prepare_dataset(
    "../data/processed/train_feature_store.parquet"
)

X = df.drop(columns=["TARGET"])

feature_names = X.columns

print(X.shape)

(307511, 207)


In [4]:
model = joblib.load(
    "../artifacts/models/lgbm_optuna_cv.pkl"
)

importance_df = get_feature_importance(
    model,
    feature_names
)

importance_df.head(20)

,feature,importance
41,EXT_SOURCE_3,261
176,pos_avg_future_installments,224
39,EXT_SOURCE_1,219
130,annuity_credit_ratio,217
122,credit_term,207
166,total_payment_amount,201
40,EXT_SOURCE_2,187
15,DAYS_BIRTH,186
8,AMT_GOODS_PRICE,174
134,bureau_total_credit,161


In [6]:
installment_v2_importance = (
    importance_df[
        importance_df["feature"].str.contains(
            "median_days_late|severe_late|recent_12m|recent_6m"
        )
    ]
    .sort_values(
        "importance",
        ascending=False
    )
)

installment_v2_importance

,feature,importance
167,recent_12m_late_ratio,67
168,recent_6m_late_ratio,50
164,severe_late_ratio,24
159,median_days_late,7
190,bb_severe_late_ratio,4
163,severe_late_count,4
198,bb_total_severe_late_count,3
200,bb_ever_severe_late,0


In [8]:
(
    importance_df[
        importance_df["feature"]
        .str.contains(
            "late|payment|installment"
        )
    ]
    .sort_values(
        "importance",
        ascending=False
    )
    .head(30)
)

,feature,importance
176,pos_avg_future_installments,241
166,total_payment_amount,201
157,installment_count,104
165,avg_payment_ratio,101
158,avg_days_late,87
162,late_payment_ratio,75
167,recent_12m_late_ratio,67
160,max_days_late,54
168,recent_6m_late_ratio,50
183,cc_total_payment,42


In [5]:
installment_df = (
    importance_df[
        importance_df["feature"].str.contains(
            "bb_"
        )
    ]
    .sort_values(
        "importance",
        ascending=False
    )
)

installment_df

,feature,importance
187,bb_record_count,63
205,bb_history_length,47
197,bb_months_since_last_late,37
194,bb_status_trend,33
193,bb_status_std,22
188,bb_late_ratio,20
203,bb_total_late_count,18
190,bb_recent_late_ratio,17
200,bb_max_status,9
195,bb_status_mean,7


In [6]:
installment_df = importance_df[importance_df["feature"].str.contains("bb_|cc_|pos_|installment_|prev_")]

installment_df.sort_values("importance", ascending=False)

,feature,importance
176,pos_avg_future_installments,224
151,prev_last_application_days,145
150,prev_avg_credit_amount,135
155,prev_avg_days_between_apps,132
149,prev_avg_application_amount,129
157,installment_count,118
154,prev_credit_growth,116
179,cc_max_balance,78
174,pos_active_contracts,75
146,prev_approved_count,74


### 📊 SON DURUM ANALİZİ

#### 🔥 En güçlü feature grupları
#### 🥇 TOP DRIVERS (çekirdek sinyal)
+ pos_avg_future_installments → 224 (çok güçlü)
+ prev_* features → 130–145 bandı
+ installment_count → 118
+ prev_credit_growth → 116

>Bunlar:

>“müşteri davranış + kredi büyüme + ödeme kapasitesi”

#### 🥈 BUREAU + CREDIT CARD ORTA GÜÇ
+ cc_max_balance → 78
+ cc_utilization_ratio → 63
+ cc_avg_credit_limit → 57
+ cc_avg_payment → 45
+ bb_record_count → 63
+ bb_history_length → 47

>Bunlar:

>“risk exposure + borç yönetimi”

#### 🥉 BUREAU BALANCE (ZAYIF AMA VAR)
+ bb_status_trend → 33
+ bb_status_std → 22
+ bb_late_ratio → 20
+ bb_total_late_count → 18


In [4]:
import joblib

from src.preprocessing.prepare_dataset import (
    prepare_dataset
)

from src.models.feature_importance import (
    get_feature_importance
)

In [5]:
df = prepare_dataset(
    "../data/processed/train_feature_store.parquet"
)

X = df.drop(columns=["TARGET"])

feature_names = X.columns

print(X.shape)

(307511, 217)


In [6]:
model = joblib.load(
    "../artifacts/models/lgbm_optuna_cv.pkl"
)

In [7]:
importance_df = get_feature_importance(
    model,
    feature_names
)

importance_df.head()

,feature,importance
41,EXT_SOURCE_3,241
122,credit_term,237
39,EXT_SOURCE_1,223
176,pos_avg_future_installments,223
130,annuity_credit_ratio,206


In [8]:
TOP_K = 150

top_features = (
    importance_df
    .head(TOP_K)["feature"]
    .tolist()
)

len(top_features)

150

In [9]:
top_features[:20]

['EXT_SOURCE_3',
 'credit_term',
 'EXT_SOURCE_1',
 'pos_avg_future_installments',
 'annuity_credit_ratio',
 'EXT_SOURCE_2',
 'DAYS_BIRTH',
 'total_payment_amount',
 'bureau_total_credit',
 'DAYS_LAST_PHONE_CHANGE',
 'DAYS_REGISTRATION',
 'AMT_GOODS_PRICE',
 'DAYS_ID_PUBLISH',
 'AMT_ANNUITY',
 'prev_last_application_days',
 'bureau_avg_credit_per_loan',
 'age_years',
 'bureau_debt_credit_ratio',
 'prev_avg_credit_amount',
 'REGION_POPULATION_RELATIVE']

In [10]:
from sklearn.model_selection import train_test_split
import lightgbm as lgb
from sklearn.metrics import roc_auc_score

In [11]:
TOP_K = 150

top_features = (
    importance_df
    .head(TOP_K)["feature"]
    .tolist()
)

X_top150 = X[top_features]

print(X_top150.shape)

(307511, 150)


In [12]:
y = df["TARGET"]

X_train, X_valid, y_train, y_valid = train_test_split(
    X_top150,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_valid.shape)

(246008, 150)
(61503, 150)


In [13]:
model = lgb.LGBMClassifier(
    n_estimators=1362,
    learning_rate=0.09460095501084868,
    num_leaves=87,
    max_depth=3,
    min_child_samples=121,
    subsample=0.7829257002191399,
    colsample_bytree=0.6053061501418525,
    reg_alpha=4.147506457250209,
    reg_lambda=2.240416755904384,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

model.fit(
    X_train,
    y_train
)

,num_leaves,87
,max_depth,3
,learning_rate,0.09460095501084868
,n_estimators,1362
,min_child_samples,121
,subsample,0.7829257002191399
,colsample_bytree,0.6053061501418525
,reg_alpha,4.147506457250209
,reg_lambda,2.240416755904384
,random_state,42
,n_jobs,-1


In [14]:
preds = model.predict_proba(X_valid)[:, 1]

auc = roc_auc_score(
    y_valid,
    preds
)

print("Top-150 ROC-AUC:", auc)

Top-150 ROC-AUC: 0.7880255637850108


#### final feature pruning (v2)

In [15]:
import numpy as np
import pandas as pd
import shap

**STEP 1 — Correlation Pruning**

In [16]:
corr_matrix = X.corr().abs()

In [17]:
upper = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

In [18]:
to_drop_corr = [
    column for column in upper.columns
    if any(upper[column] > 0.95)
]

len(to_drop_corr)

46

In [19]:
X_corr = X.drop(columns=to_drop_corr)

print(X_corr.shape)

(307511, 171)


**STEP 2 — SHAP Bottom 30 Drop**


In [20]:
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_valid)

c:\Users\ASUS\creditguard-ai\venv\Lib\site-packages\shap\explainers\_tree.py:620: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


In [21]:
shap_importance = pd.DataFrame({
    "feature": X_valid.columns,
    "mean_abs_shap": abs(shap_values).mean(axis=0)
})

shap_importance = shap_importance.sort_values(
    "mean_abs_shap",
    ascending=False
)

shap_importance.head(30)

,feature,mean_abs_shap
5,EXT_SOURCE_2,0.298233
0,EXT_SOURCE_3,0.254474
11,AMT_GOODS_PRICE,0.191679
3,pos_avg_future_installments,0.168176
2,EXT_SOURCE_1,0.152504
7,total_payment_amount,0.151797
66,CODE_GENDER,0.134132
13,AMT_ANNUITY,0.108144
4,annuity_credit_ratio,0.106976
89,NAME_EDUCATION_TYPE,0.090540


In [22]:
bottom_30 = (
    shap_importance
    .sort_values("mean_abs_shap", ascending=True)
    .head(30)["feature"]
    .tolist()
)

In [23]:
X_final = X_corr.drop(
    columns=[c for c in bottom_30 if c in X_corr.columns]
)

print(X_final.shape)

(307511, 151)


**STEP 3 — Final Dataset Check**

In [24]:
print("Before:", X.shape)
print("After corr + shap pruning:", X_final.shape)

Before: (307511, 217)
After corr + shap pruning: (307511, 151)


**STEP 4 — QUICK MODEL TEST**

In [25]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import lightgbm as lgb

In [26]:
y = df["TARGET"]

X_train, X_valid, y_train, y_valid = train_test_split(
    X_final,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [27]:
model = lgb.LGBMClassifier(
    n_estimators=1362,
    learning_rate=0.0946,
    num_leaves=87,
    max_depth=3,
    min_child_samples=121,
    subsample=0.78,
    colsample_bytree=0.60,
    reg_alpha=4.14,
    reg_lambda=2.24,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

model.fit(X_train, y_train)

,num_leaves,87
,max_depth,3
,learning_rate,0.0946
,n_estimators,1362
,min_child_samples,121
,subsample,0.78
,colsample_bytree,0.6
,reg_alpha,4.14
,reg_lambda,2.24
,random_state,42
,n_jobs,-1


In [28]:
preds = model.predict_proba(X_valid)[:, 1]

auc = roc_auc_score(y_valid, preds)

print("FINAL PRUNED ROC-AUC:", auc)

FINAL PRUNED ROC-AUC: 0.7878515414972621


In [29]:
len(X_final.columns)

151

In [30]:
list(X_final.columns)

['NAME_CONTRACT_TYPE',
 'CODE_GENDER',
 'FLAG_OWN_CAR',
 'FLAG_OWN_REALTY',
 'AMT_INCOME_TOTAL',
 'AMT_CREDIT',
 'AMT_ANNUITY',
 'NAME_INCOME_TYPE',
 'NAME_EDUCATION_TYPE',
 'NAME_FAMILY_STATUS',
 'NAME_HOUSING_TYPE',
 'REGION_POPULATION_RELATIVE',
 'DAYS_BIRTH',
 'DAYS_EMPLOYED',
 'DAYS_REGISTRATION',
 'DAYS_ID_PUBLISH',
 'OWN_CAR_AGE',
 'FLAG_MOBIL',
 'FLAG_EMP_PHONE',
 'FLAG_WORK_PHONE',
 'FLAG_CONT_MOBILE',
 'FLAG_PHONE',
 'FLAG_EMAIL',
 'OCCUPATION_TYPE',
 'REGION_RATING_CLIENT',
 'WEEKDAY_APPR_PROCESS_START',
 'HOUR_APPR_PROCESS_START',
 'REG_REGION_NOT_LIVE_REGION',
 'REG_REGION_NOT_WORK_REGION',
 'LIVE_REGION_NOT_WORK_REGION',
 'REG_CITY_NOT_LIVE_CITY',
 'REG_CITY_NOT_WORK_CITY',
 'LIVE_CITY_NOT_WORK_CITY',
 'ORGANIZATION_TYPE',
 'EXT_SOURCE_1',
 'EXT_SOURCE_2',
 'EXT_SOURCE_3',
 'APARTMENTS_AVG',
 'BASEMENTAREA_AVG',
 'COMMONAREA_AVG',
 'ELEVATORS_AVG',
 'ENTRANCES_AVG',
 'FLOORSMAX_AVG',
 'FLOORSMIN_AVG',
 'NONLIVINGAPARTMENTS_AVG',
 'NONLIVINGAREA_AVG',
 'FONDKAPREMONT_MODE'

In [31]:
for col in X_final.columns:
    print(col)

NAME_CONTRACT_TYPE
CODE_GENDER
FLAG_OWN_CAR
FLAG_OWN_REALTY
AMT_INCOME_TOTAL
AMT_CREDIT
AMT_ANNUITY
NAME_INCOME_TYPE
NAME_EDUCATION_TYPE
NAME_FAMILY_STATUS
NAME_HOUSING_TYPE
REGION_POPULATION_RELATIVE
DAYS_BIRTH
DAYS_EMPLOYED
DAYS_REGISTRATION
DAYS_ID_PUBLISH
OWN_CAR_AGE
FLAG_MOBIL
FLAG_EMP_PHONE
FLAG_WORK_PHONE
FLAG_CONT_MOBILE
FLAG_PHONE
FLAG_EMAIL
OCCUPATION_TYPE
REGION_RATING_CLIENT
WEEKDAY_APPR_PROCESS_START
HOUR_APPR_PROCESS_START
REG_REGION_NOT_LIVE_REGION
REG_REGION_NOT_WORK_REGION
LIVE_REGION_NOT_WORK_REGION
REG_CITY_NOT_LIVE_CITY
REG_CITY_NOT_WORK_CITY
LIVE_CITY_NOT_WORK_CITY
ORGANIZATION_TYPE
EXT_SOURCE_1
EXT_SOURCE_2
EXT_SOURCE_3
APARTMENTS_AVG
BASEMENTAREA_AVG
COMMONAREA_AVG
ELEVATORS_AVG
ENTRANCES_AVG
FLOORSMAX_AVG
FLOORSMIN_AVG
NONLIVINGAPARTMENTS_AVG
NONLIVINGAREA_AVG
FONDKAPREMONT_MODE
HOUSETYPE_MODE
TOTALAREA_MODE
WALLSMATERIAL_MODE
EMERGENCYSTATE_MODE
OBS_30_CNT_SOCIAL_CIRCLE
DEF_30_CNT_SOCIAL_CIRCLE
DEF_60_CNT_SOCIAL_CIRCLE
DAYS_LAST_PHONE_CHANGE
FLAG_DOCUMENT_2
FLA

In [32]:
print("\n".join(X_final.columns))

NAME_CONTRACT_TYPE
CODE_GENDER
FLAG_OWN_CAR
FLAG_OWN_REALTY
AMT_INCOME_TOTAL
AMT_CREDIT
AMT_ANNUITY
NAME_INCOME_TYPE
NAME_EDUCATION_TYPE
NAME_FAMILY_STATUS
NAME_HOUSING_TYPE
REGION_POPULATION_RELATIVE
DAYS_BIRTH
DAYS_EMPLOYED
DAYS_REGISTRATION
DAYS_ID_PUBLISH
OWN_CAR_AGE
FLAG_MOBIL
FLAG_EMP_PHONE
FLAG_WORK_PHONE
FLAG_CONT_MOBILE
FLAG_PHONE
FLAG_EMAIL
OCCUPATION_TYPE
REGION_RATING_CLIENT
WEEKDAY_APPR_PROCESS_START
HOUR_APPR_PROCESS_START
REG_REGION_NOT_LIVE_REGION
REG_REGION_NOT_WORK_REGION
LIVE_REGION_NOT_WORK_REGION
REG_CITY_NOT_LIVE_CITY
REG_CITY_NOT_WORK_CITY
LIVE_CITY_NOT_WORK_CITY
ORGANIZATION_TYPE
EXT_SOURCE_1
EXT_SOURCE_2
EXT_SOURCE_3
APARTMENTS_AVG
BASEMENTAREA_AVG
COMMONAREA_AVG
ELEVATORS_AVG
ENTRANCES_AVG
FLOORSMAX_AVG
FLOORSMIN_AVG
NONLIVINGAPARTMENTS_AVG
NONLIVINGAREA_AVG
FONDKAPREMONT_MODE
HOUSETYPE_MODE
TOTALAREA_MODE
WALLSMATERIAL_MODE
EMERGENCYSTATE_MODE
OBS_30_CNT_SOCIAL_CIRCLE
DEF_30_CNT_SOCIAL_CIRCLE
DEF_60_CNT_SOCIAL_CIRCLE
DAYS_LAST_PHONE_CHANGE
FLAG_DOCUMENT_2
FLA

In [33]:
with open("../src/feature_store/final_features.py", "w", encoding="utf-8") as f:
    f.write("FINAL_FEATURES = [\n")

    for col in X_final.columns:
        f.write(f'    "{col}",\n')

    f.write("]\n")

print("final_features.py oluşturuldu.")

final_features.py oluşturuldu.
